# Does Unemployment Increase During Recessions?

This notebook tests whether the US unemployment rate is higher, and rises faster, during NBER-dated recessions than during expansions.

**Expectation:** Yes. During economic downturns, firms hire less and lay off more workers, so unemployment should rise.

The analysis covers January 2000 to the present, using monthly data from FRED.

## Setup

The libraries used in this notebook:

- **fredapi** pulls series directly from the FRED API.
- **python-dotenv** loads the FRED API key from a local `.env` file. The file is listed in `.gitignore`, so the key never enters the repo.
- **pandas** cleans and merges the data.
- **matplotlib** draws the charts.
- **statsmodels** runs the regressions.

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
from dotenv import load_dotenv
from fredapi import Fred

## Data Pull

Two monthly series come from FRED:

| Series ID | Description | Original source |
|---|---|---|
| `UNRATE` | Civilian unemployment rate (%) | Bureau of Labor Statistics |
| `USREC` | Recession indicator (1 = recession month, 0 = not) | National Bureau of Economic Research |

The pull starts at 1999-12-01, one month before the analysis window. That extra month means the monthly change in unemployment can be computed for January 2000.

In [2]:
# Read the key from .env rather than hard-coding it, so it stays out of the repo.
load_dotenv()
api_key = os.getenv("FRED_API_KEY")
if not api_key:
    raise RuntimeError("FRED_API_KEY not found; add it to the .env file in this folder.")

fred = Fred(api_key=api_key)

START = "1999-12-01"
unrate = fred.get_series("UNRATE", observation_start=START).rename("UNRATE")
usrec = fred.get_series("USREC", observation_start=START).rename("USREC")

In [3]:
display(unrate.head())
display(usrec.head())

1999-12-01    4.0
2000-01-01    4.0
2000-02-01    4.1
2000-03-01    4.0
2000-04-01    3.8
Name: UNRATE, dtype: float64

1999-12-01    0.0
2000-01-01    0.0
2000-02-01    0.0
2000-03-01    0.0
2000-04-01    0.0
Name: USREC, dtype: float64

## Cleaning

The cleaning steps follow the plan:

1. **Merge** UNRATE and USREC into one DataFrame on date.
2. **Compute the monthly change** in the unemployment rate (this month minus last month).
3. **Drop months** where either series is missing.
4. **Convert the recession indicator** to 0/1 integers (FRED returns it as 0.0/1.0).
5. **Restrict** the data to January 2000 onward.

The change is computed on a complete monthly calendar *before* missing months are dropped. FRED has no unemployment value for October 2025 (the government shutdown delayed that month's survey). Computing the change after dropping it would make November 2025's "monthly" change span two months. Computing it first leaves November's change blank, and the averages below skip it. The extra December 1999 month from the pull is used only to compute January 2000's change.

In [ ]:
# asfreq("MS") lays the data on a full month-start calendar, so a missing month
# shows up as a gap that diff() won't bridge.
df = pd.concat([unrate, usrec], axis=1).asfreq("MS")
df["UNRATE_CHANGE"] = df["UNRATE"].diff()

df = df.dropna(subset=["UNRATE", "USREC"])
df["USREC"] = df["USREC"].astype(int)
df = df.loc["2000-01-01":]

print(f"{len(df)} months, {df.index.min():%Y-%m} to {df.index.max():%Y-%m}")
df.head()

In [ ]:
summary = (
    df.groupby("USREC")
    .agg(
        months=("UNRATE", "size"),
        avg_unemployment_rate=("UNRATE", "mean"),
        avg_monthly_change=("UNRATE_CHANGE", "mean"),
    )
    .rename(index={0: "Expansion", 1: "Recession"})
    .rename_axis("Period")
)
summary.round(2)

Since 2000, the average unemployment rate during recession months (about 6.4%) has been roughly a percentage point higher than during expansion months (about 5.5%). The monthly change shows the bigger difference: during recessions unemployment rose by about 0.6 points per month on average, while during expansions it fell slightly (about −0.06 points per month).

This supports the expectation. The averages also come from only 28 recession months across three recessions (2001, 2007–09 and 2020). The 2020 COVID spike, when unemployment jumped about 10 points in a single month, pulls the recession average up a lot. The regressions later in the notebook test whether these differences are statistically significant.